<!-- CONCLUSION-CELL -->
> ## ❌ 결론 — 1D-CNN 시퀀스 모델: **기각 (Val 기준 Aggregate XGB 우세)**
>
> **🔬 30 seed paired t-test (Val R²)**
> - Aggregate XGB **0.0800** vs 1D-CNN **0.0612**
> - 평균 차이 **-0.0188**, **t=-20.10, p=0.0000 → XGB 유의하게 우세 ✅** (30/30 전승)
>
> | 모델 | Val R² | Test R² |
> |---|---|---|
> | **Aggregate XGB** ⭐ | **0.0800** | 0.0258 |
> | 1D-CNN | 0.0612 | 0.0381 |
>
> - Val 30 seed에서 CNN이 **단 한 판도 못 이김** → 모델 선택은 명확히 XGB.
> - ⚠ Test 단일 seed에선 CNN(0.0381)>XGB(0.0258) 역전 — test는 1판이라 노이즈,
>   방법론상 **Val 30 seed 기준으로 XGB 선택**이 정당.
> - **→ 시퀀스 CNN 기각. 정형 Aggregate XGB 최종 유지.**


# 15. 시퀀스 모델 실험 (Phase 11 — 1D-CNN, 로드맵 C)

**목적**: 지금까지는 15구를 평균낸 정적(aggregate) feature를 트리에 넣었다. 여기서는 **pitch-by-pitch 시퀀스 원본**을 1D-CNN에 넣어, aggregate 트리 모델보다 예측력이 좋은지 비교한다. (로드맵 본체 — "발전시킨 것")

**입력 비교**

| 모델 | 입력 형태 | 비고 |
|---|---|---|
| Aggregate XGB (기준) | 경기당 1행 정적 feature | Phase 8 최종과 동일 split |
| **1D-CNN (제안)** | 경기당 (15구 × 구별 feature) 시퀀스 | pitch 순서 정보 보존 |

- Y: `y_whiff` (회귀, 기존과 동일)
- split: train 2021~2023 / val 2024 / test 2025 (동일)
- 비교 대상 근거: KBO 1D-CNN 헛스윙 예측 (Y 동일 계열)
- 🔬 E9: 30 seed paired t-test (XGB vs CNN Val R²)

> ⚠ **독립 실행 가능**: 이 노트북은 14번 결과와 무관하게 돌아간다(원본 pitch 시퀀스 사용). 위→아래 Run All. Colab은 런타임 GPU 권장(런타임 → 런타임 유형 변경 → T4 GPU)이지만 CPU로도 동작.

In [ ]:
# ── 환경 감지 ──────────────────────────────
import os, sys

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/투수 컨디션 예측 ML'
    NB_DIR = os.path.join(DRIVE, '3_modeling')
    sys.path.insert(0, NB_DIR)
else:
    DRIVE  = os.path.dirname(os.path.abspath('__file__'))
    NB_DIR = os.path.dirname(os.path.abspath('__file__'))

INTERIM_DIR = os.path.join(DRIVE, '0_data', '2_interim')
OUTPUT_DIR  = os.path.join(DRIVE, '4_output')
STARTERS    = os.path.join(INTERIM_DIR, 'starters_all.parquet')
LOOKUP      = os.path.join(INTERIM_DIR, 'prev_season_lookup.parquet')
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODE, N = 'pitch', 15   # X구간: 처음 15구
print(f'환경: {"코랩" if IN_COLAB else "로컬"}  |  X구간: {MODE}{N}')

In [ ]:
# ── 패키지 ─────────────────────────────────
try:
    import duckdb, torch, xgboost
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'duckdb', 'torch', 'xgboost', '-q'])

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import torch
import torch.nn as nn
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
import importlib, feature_aggregator
importlib.reload(feature_aggregator)
from feature_aggregator import build_features

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('패키지 로드 완료  |  device:', DEVICE)

## 1. 시퀀스 텐서 빌드

각 경기(game_pk × pitcher)의 처음 15구를 시간순으로 뽑아 `(경기수, 15, 구별feature)` 텐서를 만든다. 15구 미만이면 0 패딩, 결측은 0으로 채운다. Y(`y_whiff`)는 `build_features`로 가져와 키로 정렬 병합.

In [ ]:
# 구별(per-pitch) feature 정의
SEQ_FEATS = ['release_speed', 'release_spin_rate', 'release_pos_x',
             'release_pos_z', 'release_extension']

con = duckdb.connect(':memory:')
try:
    con.execute('PRAGMA disable_progress_bar')
except Exception:
    pass
con.execute(f"""
    CREATE OR REPLACE TABLE starters AS
    SELECT * FROM read_parquet('{STARTERS}')
""")

long = con.execute(f"""
    WITH ranked AS (
        SELECT game_pk, pitcher, season,
               ROW_NUMBER() OVER (PARTITION BY game_pk, pitcher
                                  ORDER BY at_bat_number, pitch_number) AS rk,
               {', '.join(SEQ_FEATS)}
        FROM starters
    )
    SELECT * FROM ranked WHERE rk <= {N} ORDER BY game_pk, pitcher, rk
""").df()
con.close()
print(f'long format: {len(long):,}행')

In [ ]:
# Y + split 키 가져오기 (aggregate 데이터와 동일한 모집단으로 맞춤)
agg = build_features(STARTERS, LOOKUP, MODE, N, include_delta=True)
KEYS = ['game_pk', 'pitcher', 'season']
y_map = agg[KEYS + ['y_whiff']].drop_duplicates()

# 시퀀스가 있는 경기만, 그리고 Y가 있는 경기만 사용
long = long.merge(y_map[KEYS], on=KEYS, how='inner')
game_index = y_map.merge(long[KEYS].drop_duplicates(), on=KEYS, how='inner').reset_index(drop=True)
print(f'사용 경기 수: {len(game_index):,}')

# (경기, 15, feat) 텐서 구성
F = len(SEQ_FEATS)
key2idx = {tuple(r): i for i, r in enumerate(game_index[KEYS].values)}
X_seq = np.zeros((len(game_index), N, F), dtype=np.float32)

vals = long[SEQ_FEATS].to_numpy(dtype=np.float32)
vals = np.nan_to_num(vals, nan=0.0)
rks  = long['rk'].to_numpy() - 1   # 0-based 시퀀스 위치
gkeys = list(map(tuple, long[KEYS].to_numpy()))
for v, rk, gk in zip(vals, rks, gkeys):
    gi = key2idx.get(gk)
    if gi is not None and 0 <= rk < N:
        X_seq[gi, rk] = v

y_all    = game_index['y_whiff'].to_numpy(dtype=np.float32)
season_all = game_index['season'].to_numpy()
print('X_seq shape:', X_seq.shape, '| y shape:', y_all.shape)

In [ ]:
# 시즌 기반 split + 정규화 (train 통계로만 fit → leakage 방지)
tr_mask = np.isin(season_all, [2021, 2022, 2023])
vl_mask = season_all == 2024
te_mask = season_all == 2025

mu  = X_seq[tr_mask].reshape(-1, F).mean(axis=0)
sd  = X_seq[tr_mask].reshape(-1, F).std(axis=0) + 1e-6
X_norm = (X_seq - mu) / sd

Xtr, ytr = X_norm[tr_mask], y_all[tr_mask]
Xvl, yvl = X_norm[vl_mask], y_all[vl_mask]
Xte, yte = X_norm[te_mask], y_all[te_mask]
print(f'train {len(Xtr):,} | val {len(Xvl):,} | test {len(Xte):,}')

## 2. 1D-CNN 정의 및 학습 함수

In [ ]:
class PitchCNN(nn.Module):
    """pitch 시퀀스 (B, F, T) → whiff% 회귀."""
    def __init__(self, n_feat, seq_len=15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_feat, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):           # x: (B, T, F)
        x = x.transpose(1, 2)        # → (B, F, T)
        return self.head(self.net(x)).squeeze(-1)


def train_cnn(Xtr, ytr, Xvl, yvl, seed=42, epochs=60, patience=8, bs=256, lr=1e-3, verbose=False):
    """early stopping 학습 → best val 기준 (val_r2, val_pred 모델) 반환."""
    torch.manual_seed(seed); np.random.seed(seed)
    model = PitchCNN(Xtr.shape[2], Xtr.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = nn.MSELoss()

    Xtr_t = torch.tensor(Xtr, device=DEVICE); ytr_t = torch.tensor(ytr, device=DEVICE)
    Xvl_t = torch.tensor(Xvl, device=DEVICE); yvl_t = torch.tensor(yvl, device=DEVICE)
    n = len(Xtr_t)

    best_r2, best_state, wait = -1e9, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            opt.zero_grad()
            loss = lossf(model(Xtr_t[idx]), ytr_t[idx])
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vpred = model(Xvl_t).cpu().numpy()
        vr2 = r2_score(yvl, vpred)
        if vr2 > best_r2:
            best_r2, best_state, wait = vr2, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience:
                break
        if verbose and ep % 10 == 0:
            print(f'  ep{ep:3d}  val_R²={vr2:.4f}')
    model.load_state_dict(best_state)
    return best_r2, model

## 3. 단일 실행 — CNN vs Aggregate XGB

In [ ]:
# CNN (seed=42)
cnn_r2, cnn_model = train_cnn(Xtr, ytr, Xvl, yvl, seed=42, verbose=True)
with torch.no_grad():
    cnn_test_pred = cnn_model(torch.tensor(Xte, device=DEVICE)).cpu().numpy()
cnn_test_r2 = r2_score(yte, cnn_test_pred) if len(Xte) else float('nan')
print(f'\n[1D-CNN]  Val R²={cnn_r2:.4f}  Test R²={cnn_test_r2:.4f}')

In [ ]:
# Aggregate XGB (동일 모집단 — game_index에 맞춘 agg)
agg_use = game_index.merge(agg, on=KEYS + ['y_whiff'], how='left')
META = ['game_pk', 'pitcher', 'season', 'y_whiff', 'swings']
FEAT = [c for c in agg_use.columns if c not in META]

a_tr, a_vl, a_te = agg_use[tr_mask], agg_use[vl_mask], agg_use[te_mask]
xgb_m = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=42, n_jobs=-1, verbosity=0,
                         early_stopping_rounds=50)
xgb_m.fit(a_tr[FEAT], a_tr['y_whiff'], eval_set=[(a_vl[FEAT], a_vl['y_whiff'])], verbose=False)
xgb_r2      = r2_score(a_vl['y_whiff'], xgb_m.predict(a_vl[FEAT]))
xgb_test_r2 = r2_score(a_te['y_whiff'], xgb_m.predict(a_te[FEAT])) if len(a_te) else float('nan')
print(f'[Aggregate XGB]  Val R²={xgb_r2:.4f}  Test R²={xgb_test_r2:.4f}')
print(f'[1D-CNN]         Val R²={cnn_r2:.4f}  Test R²={cnn_test_r2:.4f}')

## 4. 🔬 E9 Paired t-test — Aggregate XGB vs 1D-CNN

30 seed 반복. CNN은 학습이 무거우므로 시간이 걸린다(GPU 권장).

In [ ]:
N_SEEDS = 30   # 시간 부족하면 10으로 줄여도 됨
r2_xgb, r2_cnn = [], []

for i, seed in enumerate(range(N_SEEDS)):
    # XGB
    m = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=seed, n_jobs=-1, verbosity=0,
                         early_stopping_rounds=50)
    m.fit(a_tr[FEAT], a_tr['y_whiff'], eval_set=[(a_vl[FEAT], a_vl['y_whiff'])], verbose=False)
    r2_xgb.append(r2_score(a_vl['y_whiff'], m.predict(a_vl[FEAT])))
    # CNN
    cr2, _ = train_cnn(Xtr, ytr, Xvl, yvl, seed=seed)
    r2_cnn.append(cr2)
    if (i + 1) % 5 == 0:
        print(f'  {i+1}/{N_SEEDS}  (XGB {np.mean(r2_xgb):.4f} / CNN {np.mean(r2_cnn):.4f})')

r2_xgb = np.array(r2_xgb); r2_cnn = np.array(r2_cnn)
diff = r2_cnn - r2_xgb
t_stat, p_value = stats.ttest_rel(r2_cnn, r2_xgb)

print('\n' + '=' * 52)
print('Paired t-test: 1D-CNN vs Aggregate XGB')
print('=' * 52)
print(f'Aggregate XGB 평균 R²: {r2_xgb.mean():.4f} ± {r2_xgb.std():.4f}')
print(f'1D-CNN        평균 R²: {r2_cnn.mean():.4f} ± {r2_cnn.std():.4f}')
print(f'평균 차이 (CNN - XGB): {diff.mean():+.4f}')
print(f't-statistic: {t_stat:.4f}  |  p-value: {p_value:.4f}')
if p_value < 0.05:
    winner = 'CNN' if diff.mean() > 0 else 'XGB'
    print(f'→ p < 0.05: 통계적으로 유의미한 차이, 우세 모델 = {winner} ✅')
else:
    print('→ p ≥ 0.05: 두 모델 간 유의미한 차이 없음')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.boxplot([r2_xgb, r2_cnn], labels=['Aggregate\nXGB', '1D-CNN'], patch_artist=True,
           boxprops=dict(facecolor='steelblue', alpha=0.6))
ax.set_ylabel('Val R²')
ax.set_title(f'R² 분포 비교 (n={N_SEEDS} seeds)\np={p_value:.4f}')
ax.grid(alpha=0.3)
ax = axes[1]
ax.hist(diff, bins=15, color='darkorange', alpha=0.7, edgecolor='white')
ax.axvline(0, color='black', linestyle='--')
ax.axvline(diff.mean(), color='red', linewidth=1.5, label=f'mean={diff.mean():+.4f}')
ax.set_xlabel('R² 차이 (CNN - XGB)'); ax.set_ylabel('빈도')
ax.set_title('시퀀스 모델 기여 차이 분포'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sequence_model_ttest.png'), dpi=120, bbox_inches='tight')
plt.show()

## 5. 결과 저장

In [ ]:
import json
pd.DataFrame({'seed': range(N_SEEDS), 'r2_xgb': r2_xgb,
              'r2_cnn': r2_cnn, 'diff': diff}).to_csv(
    os.path.join(OUTPUT_DIR, 'sequence_model_ttest_seeds.csv'),
    index=False, encoding='utf-8-sig')

summary = {
    'agg_xgb_val_r2':  round(float(r2_xgb.mean()), 4),
    'cnn_val_r2':      round(float(r2_cnn.mean()), 4),
    'agg_xgb_test_r2': round(float(xgb_test_r2), 4),
    'cnn_test_r2':     round(float(cnn_test_r2), 4),
    'mean_diff':       round(float(diff.mean()), 4),
    't_stat':          round(float(t_stat), 4),
    'p_value':         round(float(p_value), 4),
    'significant':     bool(p_value < 0.05),
}
json.dump(summary, open(os.path.join(OUTPUT_DIR, 'sequence_model_results.json'),
                        'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('저장 완료:')
print('  4_output/sequence_model_ttest_seeds.csv')
print('  4_output/sequence_model_results.json')
print('  4_output/sequence_model_ttest.png')
print('\n요약:', summary)